In [ ]:
# config da camada bronze
# centraliza o caminho dos arquivos, os nomes das tabelas e a criação do schema
# o widget permite trocar o volume de entrada sem precisar alterar o código

try:
    import pyspark.sql.functions as F
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

# usa como padrão o volume do projeto e permite informar outro caminho na execução
dbutils.widgets.text("input_base_path", "/Volumes/workspace/default/inputs", "Pasta de entrada")
input_base_path = dbutils.widgets.get("input_base_path").rstrip("/")
if not input_base_path:
    raise ValueError("Informe o caminho da pasta de entrada no widget do Databricks.")

# relaciona cada um dos cinco arquivos à tabela bronze exigida no enunciado
csv_sources = {
    "tb_movies_info": "movies_info_TMDB_IMDB.csv",
    "tb_movies_financials": "movies_financials_IMDB_TMDB.csv",
    "tb_movies_metrics": "movies_metrics_IMDB_TMDB.csv",
    "tb_credits_and_tags": "credits_and_tags_IMDB_TMDB.csv",
    "tb_movies_reviews": "movies_reviews.csv",
}

# cria o schema sem apagar tabelas ou cargas que já existam
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [ ]:
# tabelas bronze.tb_movies_*
# lê os cinco csvs sem corrigir as sujeiras que devem ser tratadas apenas na silver
# preserva o histórico das execuções com tabelas delta gravadas em append

# mantém as colunas como texto para não transformar valores sujos em nulos durante a ingestão
# multiline preserva sinopses e comentários que possuem quebras de linha dentro das aspas
# quote e escape seguem o padrão de aspas usado nos arquivos recebidos
# permissive evita que uma linha suja interrompa toda a carga, deixando o tratamento para a silver
csv_read_options = {
    "header": "true",
    "inferSchema": "false",
    "multiLine": "true",
    "quote": '"',
    "escape": '"',
    "mode": "PERMISSIVE",
    "encoding": "UTF-8",
}

# registra o momento da carga para permitir rastreabilidade e deduplicação na silver
ingestion_timestamp = F.current_timestamp()

# percorre o mapeamento para aplicar a mesma regra de ingestão aos cinco arquivos
for table_name, file_name in csv_sources.items():
    source_path = f"{input_base_path}/{file_name}"

    (
        spark.read
        .options(**csv_read_options)
        .csv(source_path)
        .withColumn("ingestion_datetime", ingestion_timestamp)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(f"bronze.{table_name}")
    )

print(f"Carga concluída: {len(csv_sources)} tabelas foram gravadas na camada Bronze.")

In [ ]:
# validação das tabelas bronze de arquivos
# confirma que cada destino foi publicado em delta e recebeu a coluna de ingestão

# relê as tabelas persistidas para validar o resultado da gravação, não apenas o dataframe em memória
validation_dfs = []
for table_name in csv_sources:
    table_df = spark.table(f"bronze.{table_name}")
    detail_df = (
        spark.sql(f"DESCRIBE DETAIL bronze.{table_name}")
        .select(
            F.lit(table_name).alias("table_name"),
            F.lit("ingestion_datetime" in table_df.columns).alias("has_ingestion_datetime"),
            F.col("format").alias("storage_format"),
        )
    )
    validation_dfs.append(detail_df)

validation_result = validation_dfs[0]
for validation_df in validation_dfs[1:]:
    validation_result = validation_result.unionByName(validation_df)

# reúne as cinco verificações em uma visualização simples para facilitar a conferência
display(validation_result)

In [ ]:
# tabela bronze.tb_cotacao_dolar: período da ptax
# parametriza as datas no formato exigido pelo banco central
# o período padrão usa sete dias corridos para incluir dias úteis mesmo após feriados ou fins de semana

from datetime import date, datetime, timedelta
import requests
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# define uma janela inclusiva de sete dias terminando na data da execução
default_data_fim = date.today()
default_data_inicio = default_data_fim - timedelta(days=6)
dbutils.widgets.text("data_inicio", default_data_inicio.strftime("%m-%d-%Y"), "Data inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", default_data_fim.strftime("%m-%d-%Y"), "Data final (MM-DD-AAAA)")

data_inicio = datetime.strptime(dbutils.widgets.get("data_inicio"), "%m-%d-%Y").date()
data_fim = datetime.strptime(dbutils.widgets.get("data_fim"), "%m-%d-%Y").date()
if data_inicio > data_fim:
    raise ValueError("A data inicial deve ser menor ou igual à data final.")

# consulta também dias anteriores para fornecer uma cotação-semente ao forward fill da silver
data_inicio_consulta_api = data_inicio - timedelta(days=7)

print(f"Período solicitado: {data_inicio:%m-%d-%Y} a {data_fim:%m-%d-%Y}")
print(f"Período consultado na API: {data_inicio_consulta_api:%m-%d-%Y} a {data_fim:%m-%d-%Y}")

In [ ]:
# tabela bronze.tb_cotacao_dolar: consulta da ptax
# consulta o endpoint oficial e registra junto da resposta o período solicitado
# mantém o histórico das consultas em delta e append como nas demais tabelas bronze

api_url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)
# envia somente a cotação de compra e o horário necessários para a conversão em reais
request_params = {
    "@dataInicial": f"'{data_inicio_consulta_api:%m-%d-%Y}'",
    "@dataFinalCotacao": f"'{data_fim:%m-%d-%Y}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json",
}
# o timeout e as validações fazem o job falhar de forma clara quando a api não responde corretamente
response = requests.get(api_url, params=request_params, timeout=30)
if response.status_code != 200:
    raise RuntimeError(f"Erro ao consultar a API do Banco Central: {response.status_code} - {response.text}")

cotacoes = response.json().get("value", [])
if not cotacoes:
    raise ValueError("A API do Banco Central não retornou cotações para o período consultado.")
cotacao_schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True),
])
# transforma a resposta em dataframe e guarda os limites usados para orientar a silver
df_cotacao_dolar = (
    spark.createDataFrame(cotacoes, schema=cotacao_schema)
    .withColumn("data_inicio_periodo", F.lit(data_inicio.isoformat()).cast("date"))
    .withColumn("data_fim_periodo", F.lit(data_fim.isoformat()).cast("date"))
    .withColumn("data_inicio_consulta_api", F.lit(data_inicio_consulta_api.isoformat()).cast("date"))
    .withColumn("ingestion_datetime", F.current_timestamp())
)

# grava a resposta sem substituir consultas anteriores
(
df_cotacao_dolar.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("bronze.tb_cotacao_dolar")
)

print(f"Coleta de dados concluída: {df_cotacao_dolar.count()} cotações gravadas em bronze.tb_cotacao_dolar.")

In [ ]:
# validação de bronze.tb_cotacao_dolar
# confirma o formato delta e os campos necessários para conversão e forward fill

# valida a tabela persistida para garantir que a resposta foi realmente publicada
cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")
validation_cotacao = (
    spark.sql("DESCRIBE DETAIL bronze.tb_cotacao_dolar")
    .select(
        F.lit("tb_cotacao_dolar").alias("table_name"),
        F.lit("ingestion_datetime" in cotacao_bronze.columns).alias("has_ingestion_datetime"),
        F.lit("dataHoraCotacao" in cotacao_bronze.columns).alias("has_dataHoraCotacao"),
        F.lit("cotacaoCompra" in cotacao_bronze.columns).alias("has_cotacaoCompra"),
        F.lit("data_inicio_periodo" in cotacao_bronze.columns).alias("has_data_inicio_periodo"),
        F.lit("data_fim_periodo" in cotacao_bronze.columns).alias("has_data_fim_periodo"),
        F.lit("data_inicio_consulta_api" in cotacao_bronze.columns).alias("has_data_inicio_consulta_api"),
        F.col("format").alias("storage_format"),
    )
)

display(validation_cotacao)